In [1]:
#!pip install pyvis

In [2]:
import networkx as nx
import pandas as pd
from pyvis.network import Network
from community import community_louvain
from collections import defaultdict



# Load your data
data = pd.read_csv('inek.csv')

# Create a NetworkX graph
G = nx.from_pandas_edgelist(data, 'Source', 'Target', edge_attr=True)

# Initialize node importance based on occurrences
node_importance = {node: 0 for node in G.nodes()}

# Update the node importance based on unique Source_Occurrence and Target_Occurrence values
for _, row in data.iterrows():
    if row['Source'] in node_importance:
        node_importance[row['Source']] = row['Source_Occurrence']
    if row['Target'] in node_importance:
        node_importance[row['Target']] = row['Target_Occurrence']

# Min-max normalization
max_importance = max(node_importance.values())
min_importance = min(node_importance.values())
scaling_factor = 40
threshold = 0.005  # Set a small threshold to filter near-zero importance nodes

for node in node_importance:
    if max_importance != min_importance:  # To avoid division by zero
        node_importance[node] = ((node_importance[node] - min_importance) /
                                 (max_importance - min_importance)) * scaling_factor
    else:
        # If all values are the same, set them to a default normalized value
        node_importance[node] = 1

# Filter out nodes below the importance threshold
filtered_nodes = {node: size for node, size in node_importance.items() if size > threshold}

# Detect communities
partition = community_louvain.best_partition(G)

# Filter partition to only include nodes that passed the filtering
filtered_partition = {node: comm for node, comm in partition.items() if node in filtered_nodes}

# Community leaders remain for analysis purposes
community_leader = {}
for node, comm in partition.items():
    if node in filtered_nodes and (comm not in community_leader or node_importance[node] > node_importance[community_leader[comm]]):
        community_leader[comm] = node

print("Community Leaders:")
print(community_leader)

# Define a dictionary to map color codes to color names
color_dict = {
    '#FF0000': 'Red',
    '#ffdd00': 'Yellow',
    '#00FFFF': 'Cyan',
    '#00FF00': 'Green',
    '#ed4ed0': 'Pink',
    '#0335fc': 'Blue',
    '#8900d3': 'Purple',
    '#f296a2': 'Light Pink'
}
# Create a list of color codes
color_list = list(color_dict.keys())

# Create community colors with the reordered color list
community_colors = {com: color_list[i % len(color_list)] for i, com in enumerate(set(filtered_partition.values()))}

# Print community colors with names and community leaders
for com, color in community_colors.items():
    leader = community_leader[com]
    print(f"Theme/cluster: {leader}: {color_dict[color]}")

# Dictionary to hold clusters with filtered nodes
clusters = defaultdict(list)

# Populate the dictionary with nodes grouped by their community and sorted by importance
for node, comm in filtered_partition.items():
    clusters[comm].append((node, node_importance[node]))

# Sort each cluster's nodes by importance in descending order
for comm in clusters:
    clusters[comm].sort(key=lambda x: x[1], reverse=True)  # Sort by node importance

# Print each cluster and its nodes using community leader names
for comm in sorted(clusters.keys()):
    leader = community_leader[comm]
    print(f"Cluster: {leader} (Color: {color_dict[community_colors[comm]]}):")
    for node, importance in clusters[comm]:
        print(f"  Node: {node}, Node_Weight: {importance:.2f}")
    print()  # Add a newline for better separation between clusters

# Initialize PyVis network with ForceAtlas2 and improved label visibility
net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", notebook=True)

# Set options with adjusted label visibility
options = """
{
    "nodes": {
        "font": {
            "size": 16,
            "face": "arial"
        },
        "scaling": {
            "label": {
                "enabled": true,
                "min": 8,
                "max": 30
            }
        }
    },
    "edges": {
        "color": {
            "inherit": true
        },
        "smooth": false
    },
    "physics": {
        "forceAtlas2Based": {
            "gravitationalConstant": -1500,
            "centralGravity": 0.1,
            "springLength": 100,
            "springStrength": 0.08,
            "damping": 0.4
        },
        "minVelocity": 0.75,
        "solver": "forceAtlas2Based"
    },
    "interaction": {
        "tooltipDelay": 200,
        "hideEdgesOnDrag": false
    },
    "configure": {
        "enabled": true,
        "filter": "physics"
    }
}
"""
net.set_options(options)

# Add filtered nodes to the PyVis network
for node, size in filtered_nodes.items():
    net.add_node(node, title=node, size=size, color=community_colors[filtered_partition[node]])

# Add edges only between nodes present in the filtered nodes list
for source, target, weight in G.edges(data=True):
    if source in filtered_nodes and target in filtered_nodes:
        net.add_edge(source, target, value=weight['Edge_Weight'])

# Display the network
net.show("network.html")

Community Leaders:
{0: 'hernieuwbare', 1: 'nederland', 4: 'energie', 3: 'klimaatakkoord', 5: 'maatregelen', 6: 'co', 2: 'investeringen', 7: 'emissies'}
Theme/cluster: hernieuwbare: Red
Theme/cluster: nederland: Yellow
Theme/cluster: investeringen: Cyan
Theme/cluster: klimaatakkoord: Green
Theme/cluster: energie: Pink
Theme/cluster: maatregelen: Blue
Theme/cluster: co: Purple
Theme/cluster: emissies: Light Pink
Cluster: hernieuwbare (Color: Red):
  Node: hernieuwbare, Node_Weight: 20.00
  Node: gebouwde, Node_Weight: 9.25
  Node: aardgas, Node_Weight: 7.63
  Node: aandeel, Node_Weight: 7.10
  Node: komt, Node_Weight: 5.91
  Node: ontwikkelingen, Node_Weight: 5.81
  Node: verduurzaming, Node_Weight: 5.38
  Node: gebouwen, Node_Weight: 3.98
  Node: woningen, Node_Weight: 3.33
  Node: cbs, Node_Weight: 2.90
  Node: warmte, Node_Weight: 2.90
  Node: juni, Node_Weight: 2.58
  Node: laagcalorisch, Node_Weight: 2.58
  Node: bevat, Node_Weight: 2.37
  Node: indicatoren, Node_Weight: 1.94
  Node